In [3]:
import torch
from IPython import display
from d2l import torch as d2l

batch_size = 256
# train_iter, test_iter 表示训练集和测试集的一个迭代器
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)


将展平每个图像，将它们视为长度为784的向量。
因为数据集有10个类别，所以网络输出维度为10

In [4]:
num_inputs = 784
num_outputs = 10

# softmax需要向量。所以需要将图片拉长
# 均值为0，方差0.01， 形状为输入个数784， 输出个数10
W = torch.normal(0, 0.01, size=(num_inputs, num_outputs),requires_grad=True)
b = torch.zeros(num_outputs, requires_grad=True)

给定一个矩阵X 对所有元素求和

In [7]:
# 矩阵 2x3 
X = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
X.sum(0, keepdim=True), X.sum(1, keepdim=True) # 变成2x1 的列向量

(tensor([[5., 7., 9.]]),
 tensor([[ 6.],
         [15.]]))

实现softmax 

In [9]:
# X是矩阵，按每一行进行softmax
def softmax(X):
    X_exp = torch.exp(X)
    # 按第一行进行求和 保持2维矩阵
    partition = X_exp.sum(1, keepdim=True)
    return X_exp / partition # 使用广播机制

验证一下softmax
将每个元素变成一个非负数，此外，根据概率原理，每行总和为1

In [10]:
# 0-1, 2raw 5col matrix
x = torch.normal(0, 1, (2, 5))
x_prob = softmax(x)
x_prob, x_prob.sum(1)

(tensor([[0.3026, 0.0712, 0.0571, 0.1953, 0.3738],
         [0.0686, 0.6141, 0.1339, 0.0541, 0.1293]]),
 tensor([1.0000, 1.0000]))

实现softmax回归模型

In [28]:
def net(X):
    return softmax(torch.matmul(X.reshape((-1, W.shape[0])), W) + b)

在实现交叉熵之前补充细节
创建一个数据y_hat
其中包括2个样本，3个类别的预测概率，使用y作为y_hat中概率的索引
下面看不懂来看这个：https://www.zhihu.com/question/23765351/answer/2144591565

In [29]:
# y[0] 0  y[1] 2
y = torch.tensor([0,2])  
# y_hat是预测值 第一个图片是苹果概率为0.1 橘子0.3 香蕉0.6
y_hat = torch.tensor([[0.1, 0.3, 0.6], [0.3, 0.2, 0.5]])
# 返回y_hat第0行第y[0]列 和第一行第y[1]列
# torch.tensor([0,2]) y[0]是第0列 y[1]是第2列
# y[0,1]= 0.1  y[1,2]=0.5
# [0.1, 0.5] 拿出预测值
y_hat[[0, 1], y]

tensor([0.1000, 0.5000])

实现交叉熵损失函数

In [30]:
def cross_entropy(y_hat, y):
    return -torch.log(y_hat[range(len(y_hat)), y])

cross_entropy(y_hat, y)

tensor([2.3026, 0.6931])

将预测类别与真实y元素进行比较

In [31]:
def accuracy(y_hat, y):
    """计算预测正确"""
    # 是2维矩阵，且列数>1
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())

accuracy(y_hat, y) / len(y) 

0.5

评估在任意模型net的准确率

In [32]:
def evaluate_accuracy(net, data_iter):
    """计算在指定数据集上模型的精度"""
    if isinstance(net, torch.nn.Module):
        net.eval()
    metric = Accumulator(2) # 累加器 正确预测数，预测总数
    for X, y in data_iter:
        metric.add(accuracy(net(X), y), y.numel())
    return metric[0] / metric[1] # 返回正确的/总数

In [36]:
class Accumulator:
    """在n个变量上累加"""
    def __init__(self, n):
        self.data = [0.0] * n
    
    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]
        
    def reset(self):
        self.data = [0.0] * len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

evaluate_accuracy(net, test_iter)

0.0944

Softmax回归的训练

In [64]:
def train_epoch_ch3(net, train_iter, loss, updater):
    if isinstance(net, torch.nn.Module):
        net.train()
    metric = Accumulator(3)
    for X, y in train_iter:
        y_hat = net(X)
        l = loss(y_hat, y)
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            updater.step()
            metric.add(float(l) * len(y),
                      accuracuy(y_hat, y),
                      y.size().numel())
        else:
            l.sum().backward() # 向量求和算梯度
            updater(X.shape[0])
            metric.add(float(l.sum()),accuracy(y_hat, y), y.numel())
    return metric[0] / metric[2], metric[1] / metric[2]


训练函数

In [65]:
def train_ch3(net, train_iter, test_iter, loss, num_epochs, updater):
    for epoch in range(num_epochs):
        # 扫一次 训练的误差拿出
        train_metrics = train_epoch_ch3(net, train_iter, loss, updater)
        test_acc = evaluate_accuracy(net, test_iter)
    train_loss, train_acc = train_metrics

小批量随机梯度下降来优化模型损失函数

In [66]:
lr = 0.1

def updater(batch_size):
    return d2l.sgd([W, b], lr, batch_size)

训练模型10个迭代周期

In [67]:
num_epoches = 10
train_ch3(net, train_iter, test_iter, cross_entropy, num_epoches, updater)


KeyboardInterrupt: 

有点问题，但是大致思路没问题